In [0]:
from pyspark.sql.functions import col, count, countDistinct, sum, to_date

SILVER_PATH = "/Volumes/workspace/customer_analytics/data_files/silver/customer_events"
GOLD_PATH = "/Volumes/workspace/customer_analytics/data_files/gold"

silver_df = spark.read.format("delta").load(SILVER_PATH)

In [0]:
gold_daily_revenue = (
    silver_df
    .filter(col("event_type") == "payment_success")
    .withColumn("event_date", to_date(col("event_timestamp")))
    .groupBy("event_date")
    .agg(
        sum("price").alias("total_revenue"),
        count("*").alias("successful_payments")
    )
)

gold_daily_revenue.write.format("delta").mode("overwrite").save(
    f"{GOLD_PATH}/daily_revenue"
)

display(gold_daily_revenue)

event_date,total_revenue,successful_payments
2026-06-19,43228.48999999999,176


In [0]:
gold_user_activity = (
    silver_df
    .withColumn("event_date", to_date(col("event_timestamp")))
    .groupBy("event_date")
    .agg(
        countDistinct("user_id").alias("active_users"),
        count("*").alias("total_events")
    )
)

gold_user_activity.write.format("delta").mode("overwrite").save(
    f"{GOLD_PATH}/user_activity"
)

display(gold_user_activity)

event_date,active_users,total_events
2026-06-19,948,1000


In [0]:
gold_event_funnel = (
    silver_df
    .groupBy("event_type")
    .agg(
        count("*").alias("event_count")
    )
    .orderBy(col("event_count").desc())
)

gold_event_funnel.write.format("delta").mode("overwrite").save(
    f"{GOLD_PATH}/event_funnel"
)

display(gold_event_funnel)

event_type,event_count
payment_success,176
payment_failed,171
cart_add,170
checkout,164
product_view,162
user_login,157


In [0]:
gold_failed_payments = (
    silver_df
    .filter(col("event_type") == "payment_failed")
    .withColumn("event_date", to_date(col("event_timestamp")))
    .groupBy("event_date")
    .agg(
        count("*").alias("failed_payment_count"),
        sum("price").alias("failed_payment_value")
    )
)

gold_failed_payments.write.format("delta").mode("overwrite").save(
    f"{GOLD_PATH}/failed_payments"
)

display(gold_failed_payments)

event_date,failed_payment_count,failed_payment_value
2026-06-19,171,44023.429999999986


In [0]:
display(spark.read.format("delta").load(f"{GOLD_PATH}/daily_revenue"))
display(spark.read.format("delta").load(f"{GOLD_PATH}/user_activity"))
display(spark.read.format("delta").load(f"{GOLD_PATH}/event_funnel"))
display(spark.read.format("delta").load(f"{GOLD_PATH}/failed_payments"))

event_date,total_revenue,successful_payments
2026-06-19,43228.48999999999,176


event_date,active_users,total_events
2026-06-19,948,1000


event_type,event_count
payment_success,176
payment_failed,171
cart_add,170
checkout,164
product_view,162
user_login,157


event_date,failed_payment_count,failed_payment_value
2026-06-19,171,44023.429999999986
